# 03 - TDL2048+ / CPU / 4x4
First-stage 8x6 OTD+TC. Existing game weights are unchanged. Use a hosted Colab CPU runtime, not a local kernel.
Upload `training/tdl2048/run.py` manually to Drive: `MyProjects/2048-ai/scripts/tdl2048/run.py`. No upload widget is needed.
This notebook is not the complete two-stage 72% configuration. Stop after the initial 1000-game trial and inspect speed/evaluation before running longer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys
ROOT = Path('/content/drive/MyDrive/MyProjects/2048-ai')
SCRIPT = ROOT / 'scripts/tdl2048/run.py'
RUN = ROOT / 'models/03-otd-8x6'
WORK = Path('/content/tdl2048-build')
assert SCRIPT.is_file(), f'Upload run.py to {SCRIPT}'
def tdl(command, *options):
    args = [sys.executable, '-u', str(SCRIPT), command, '--run', str(RUN), '--work', str(WORK), *map(str, options)]
    process = subprocess.Popen(args, start_new_session=True)
    try:
        code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
    if code:
        raise RuntimeError(f'Trainer exited with code {code}; see output above')
print('CPUs:', os.cpu_count(), '\nCheckpoints:', RUN)
subprocess.run(['free', '-h'], check=True)
subprocess.run(['df', '-h', '/content'], check=True)

## Build
Run again after a new Colab runtime. No model is initialized by this cell. If GCC/Make is missing, install `build-essential` before retrying.

In [ ]:
tdl('setup')

## Small engine check
20 games with a tiny separate network: checks TD, TC, save/load and evaluation before allocating the real model. Re-running an already completed smoke run does not train more games.

In [ ]:
SMOKE = ROOT / 'outputs/tdl2048-smoke'
for _ in range(2):
    tdl('train', '--run', SMOKE, '--network', '2x4patt', '--plan-episodes', 20, '--episodes', 10, '--chunk', 5)
tdl('eval', '--run', SMOKE, '--games', 2)

## Train / resume
EPISODES is additional games this run. Re-running continues the same model. Start with 1000; after checking results use 100000. The global schedule remains 100 million episodes.
Only two complete checkpoints are kept. Stop can lose the unfinished batch; Drive synchronization is not a guarantee against every disconnect. Never run two trainers against this directory.

In [ ]:
EPISODES = 1000
CHUNK = 1000
THREADS = 2
tdl('train', '--episodes', EPISODES, '--chunk', CHUNK, '--threads', THREADS)

In [ ]:
tdl('status')

## Evaluate
Fresh boards, separate seed from training, no weight updates. Begin with depth 1. Depth 3 is stronger but slower. These are binary upstream weights, not a replacement for ntupleWeights.js.

In [ ]:
tdl('eval', '--games', 100, '--depth', 1)

After disconnect: run mount, config, build, then train again. The latest complete model and learning state are loaded automatically. Training logs: RUN/train.log. Evaluation logs: RUN/eval-*.log. Download an entire checkpoint folder, including model.w and state.json, for local resume.